In [1]:
import torch
import soundfile as sf
import numpy as np
from transformers import Wav2Vec2Model
import torch.nn as nn

DEVICE = torch.device("cpu")

# -------------------------
# MODEL CLASS (same as training)
# -------------------------
class AudioDeepfakeModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.wav2vec = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base")
        self.classifier = nn.Linear(768, 2)

    def forward(self, x):
        outputs = self.wav2vec(x)
        pooled = outputs.last_hidden_state.mean(dim=1)
        return self.classifier(pooled)

# -------------------------
# LOAD MODEL
# -------------------------
model = AudioDeepfakeModel().to(DEVICE)
model.load_state_dict(torch.load("ai_models/audio/wav2vec_audio.pth", map_location=DEVICE))
model.eval()

# -------------------------
# PREPROCESS FUNCTION
# -------------------------
def preprocess_audio(file_path):
    waveform, sr = sf.read(file_path)

    waveform = torch.tensor(waveform).float()

    # stereo → mono
    if len(waveform.shape) > 1:
        waveform = waveform.mean(dim=1)

    # fix length (3 sec)
    max_len = 16000 * 3
    if waveform.shape[0] > max_len:
        waveform = waveform[:max_len]
    else:
        waveform = torch.nn.functional.pad(waveform, (0, max_len - waveform.shape[0]))

    return waveform.unsqueeze(0)

# -------------------------
# PREDICTION FUNCTION
# -------------------------
def predict_audio(file_path):
    input_audio = preprocess_audio(file_path).to(DEVICE)

    with torch.no_grad():
        output = model(input_audio)
        probs = torch.softmax(output, dim=1)

        fake_prob = probs[0][1].item()
        real_prob = probs[0][0].item()

    print("\n🎧 Prediction Results:")
    print(f"Real Probability: {real_prob:.4f}")
    print(f"Fake Probability: {fake_prob:.4f}")

    if fake_prob > real_prob:
        print("🚨 RESULT: FAKE AUDIO")
    else:
        print("✅ RESULT: REAL AUDIO")


# -------------------------
# TEST AUDIO FILE
# -------------------------
audio_path = r"C:\Users\vansh\Downloads\WhatsApp Ptt 2026-03-16 at 7.35.21 PM.ogg"
predict_audio(audio_path)   # put your file here

C:\Users\vansh\AppData\Roaming\Python\Python313\site-packages\transformers\configuration_utils.py:335: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(



🎧 Prediction Results:
Real Probability: 0.9972
Fake Probability: 0.0028
✅ RESULT: REAL AUDIO
